# Weak-estimates playground — exposed DSL, training, pairwise, metrics

Unlike `colab_weak_estimates_sweep.ipynb` (a thin shell around a fixed
grid), everything here is in the open and editable cell by cell:

1. **Estimate list** — built with the estimate DSL (`ExpectationEstimate`,
   `ProbabilityEstimate`, `EquationEstimate`, ...): the deterministic
   links are written out literally, the weak 5-sample pool is drawn in
   front of you and printed, and there's a slot for hand-written extras.
2. **Training** — the `DistributionBuilder` construction and `fit()` call
   are exposed, so you can turn the knobs (`steps`, `n_components`,
   `eqn_conf`, ...) and re-run just that cell.
3. **Pairwise plot** — the fitted joint over any variable subset.
4. **Metrics** — KL to the closed-form truth, moment errors, link
   fidelity, and the worst-fitted constraints.

The true world is the same fixed bivariate normal as the sweep:
mean (0.5, -0.5), sds (1.0, 1.5), rho = 0.6.

**Runtime -> GPU**, then run top to bottom; iterate on cells 3-7 freely.

In [ ]:
# --- Setup: clone-or-pull the repo, install deps Colab lacks ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"
URL = "https://github.com/amdson/calibrated_response.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
                   check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

%pip -q install optax jaxopt pydantic

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"
print("setup OK")

In [ ]:
# --- Imports: the truth, the DSL, the builder --------------------------------
import numpy as np

# the true world + honest-weighting helpers (oracle sds, true event probs)
from benchmarks.weak_estimates_experiment import (
    MU, COV, RHO, SDS, N_PER_EST, ORACLE, MENU,
    build_variables, draw_pool, sample_true, true_p,
    kl_gauss, kl_knn, kl_uniform_ref, _true_logpdf, _TRANSFORMS, _LINKS)

# the estimate DSL
from calibrated_response.models.query import (
    CorrelationEstimate, EquationEstimate, ExpectationEstimate,
    InequalityProposition, ProbabilityEstimate)

from calibrated_response.maxent_sampler.distribution_builder import (
    DistributionBuilder)
from calibrated_response.maxent_sampler import plot_pairwise

print("variables:", [v.name for v in build_variables()])
print(f"true: E[x]={MU[0]} E[y]={MU[1]} sd=({SDS[0]}, {SDS[1]}) rho={RHO}")
print(f"uniform-box anchor: KL = {kl_uniform_ref():.3f} nats")

In [ ]:
# --- The estimate list, in the DSL -------------------------------------------
# (a) deterministic links: composite functionals need auxiliary variables,
#     each pinned to (x, y) by an exact EquationEstimate.
links = [
    EquationEstimate(id="def_xx",  lhs="xx",  rhs="x * x"),
    EquationEstimate(id="def_yy",  lhs="yy",  rhs="y * y"),
    EquationEstimate(id="def_xy",  lhs="xy",  rhs="x * y"),
    EquationEstimate(id="def_xpy", lhs="xpy", rhs="x + y"),
    EquationEstimate(id="def_xmy", lhs="xmy", rhs="x - y"),
]

# (b) the weak pool: M estimates, each a fresh 5-point empirical statistic
#     with its honest sampling sd (expectations: value units; probabilities:
#     log-odds width 1/sqrt(n p (1-p)), Jeffreys-smoothed p-hat).
M, SEED = 32, 0
pool = draw_pool(M, seed=SEED)

# (c) hand-written extras — this is the cell to play in.  The STRING DSL
#     covers everything: expression subjects constrain the (x, y) joint
#     directly (bypassing the aux links), and a trailing `~ w` states your
#     uncertainty about the estimate itself (fills Estimate.sd — value
#     units for E, log-odds for P, correlation units for Corr; omit for
#     the solver default).
from calibrated_response.models.natural_response import parse_natural_syntax as est

extras = [
    # est("E[x * y] = 0.65 ~ 0.37"),        # E[XY] at n=5 strength
    # est("P(x - y > 0) = 0.80 ~ 0.9"),     # P(X > Y), log-odds width
    # est("P(x > 2) = 0.07 ~ 0.9"),
    # est("Corr(x, y) = 0.6 ~ 0.2"),        # rho directly
    # est("xmy = x - y"),                   # a deterministic link, as a string
]

ESTIMATES = links + pool + extras
for e in ESTIMATES:
    sd = f"  sd={e.sd:.3f}" if getattr(e, "sd", None) is not None else ""
    print(f"{e.id:>16}  {e.to_query_estimate()}{sd}")

In [ ]:
# --- Training, exposed --------------------------------------------------------
# Knobs that matter here: steps (anneal staircase lives inside), n_components
# (K = mixture components in the flow base), eqn_conf (strength of the
# deterministic links' noisy siblings — links themselves are exact).
STEPS = 3000
K = 1

builder = DistributionBuilder(build_variables(), ESTIMATES, n_components=K)
assert not builder.skipped, builder.skipped
if builder.warnings:
    print("warnings:", *builder.warnings, sep="\n  ")

builder.fit(steps=STEPS, lr=2e-3, n_samples=2048, seed=SEED)
s = builder.sample_dict(30_000, seed=SEED + 1)
pts = np.stack([s["x"], s["y"]], axis=1)
print(f"fitted; {len(builder.constraints)} constraints, "
      f"{pts.shape[0]} eval samples")

In [ ]:
# --- Pairwise (corner) plot ----------------------------------------------------
# Any subset of ["x", "y", "xx", "yy", "xy", "xpy", "xmy"]; dashed lines mark
# the TRUE means.  xy hugging a curved ridge over (x, y) = links holding.
NAMES = ["x", "y", "xy", "xmy"]
sites = [builder.var_name_to_idx[n] for n in NAMES]
plot_pairwise(builder.model, builder.params, sites=sites, names=NAMES,
              n_samples=30_000, seed=11, bins=50,
              threshold={n: ORACLE[n][0] for n in NAMES});

In [ ]:
# --- Fit vs true contours -------------------------------------------------------
import matplotlib.pyplot as plt

gx, gy = np.meshgrid(np.linspace(-3.5, 4.5, 120), np.linspace(-6, 5, 120))
gz = np.exp(_true_logpdf(np.stack([gx.ravel(), gy.ravel()], 1))).reshape(gx.shape)
plt.figure(figsize=(6, 5))
plt.hist2d(pts[:, 0], pts[:, 1], bins=80, cmap="viridis")
plt.contour(gx, gy, gz, levels=6, colors="w", linewidths=0.8, alpha=0.8)
plt.plot(*MU, "r+", ms=14, mew=2)
plt.xlabel("x"); plt.ylabel("y")
plt.title(f"M={M} estimates, K={K}, {STEPS} steps")
plt.tight_layout(); plt.show()

In [ ]:
# --- Metrics ---------------------------------------------------------------------
mq, cq = pts.mean(axis=0), np.cov(pts.T)
link_err = float(np.mean(
    [np.sqrt(np.mean((s[v] - _TRANSFORMS[v](pts)) ** 2)) / ORACLE[v][1]
     for v in _LINKS]))

print(f"kl_gauss = {kl_gauss(pts):.4f}   kl_knn = {kl_knn(pts):.4f} "
      f"  (uniform anchor {kl_uniform_ref():.2f})")
print(f"E[x]  fit {mq[0]:+.3f}  true {MU[0]:+.3f}")
print(f"E[y]  fit {mq[1]:+.3f}  true {MU[1]:+.3f}")
print(f"sd(x) fit {np.sqrt(cq[0, 0]):.3f}  true {SDS[0]:.3f}")
print(f"sd(y) fit {np.sqrt(cq[1, 1]):.3f}  true {SDS[1]:.3f}")
print(f"corr  fit {cq[0, 1] / np.sqrt(cq[0, 0] * cq[1, 1]):+.3f}  "
      f"true {RHO:+.3f}")
print(f"link_err = {link_err:.3f}  (RMS(aux - f(x,y)) in sd(f) units)")

# worst-fitted constraints — where does residual KL live?
rep = sorted(builder.constraint_report(), key=lambda c: -abs(c["error_rel"]))
print("\nworst-fitted constraints:")
for c in rep[:12]:
    print(f"{c['id']:>16}  target={c['target']:.3f} "
          f"fitted={c['fitted']:.3f}  err_rel={c['error_rel']:+.3f}")